# Modulo 6 — EA Generator (MQL4 / MQL5)

Esporta le strategie validate come **Expert Advisor compilabili** per MetaTrader 4 e 5. Il modello (StandardScaler + centroidi KMeans) è **embeddato nel sorgente**: l'EA è autosufficiente, ricalcola le stesse feature con indicatori nativi, le standardizza, trova il cluster più vicino e — se coincide col pattern — apre il trade con SL/TP/BE/trailing in ATR e rischio percentuale.

**Vincolo di fedeltà:** solo feature calcolabili nativamente in MetaTrader sono esportabili. Per generare EA fedeli, la Pattern Discovery va eseguita su `EXPORTABLE_FEATURES`.

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine
from trading_ai.pattern_discovery import PatternDiscovery
from trading_ai.strategy_generator import StrategyGenerator, RiskParams
from trading_ai.ea_generator import EAGenerator, EXPORTABLE_FEATURES
print('Feature esportabili in MQL:', EXPORTABLE_FEATURES)

## 1. Discovery ristretta alle feature esportabili

Passiamo `feature_columns=EXPORTABLE_FEATURES` così il modello userà solo input ricalcolabili in MetaTrader.

In [ ]:
eng = DataEngine()
h1 = eng.to_timeframe(eng.load_dataframe(generate_ohlcv(n=200_000, seed=3)), 'H1')
feats = FeatureEngine().transform(h1, groups=['indicator', 'volatility'], dropna=True)
cols = [c for c in EXPORTABLE_FEATURES if c in feats.columns]
disc = PatternDiscovery(n_clusters=20, horizon=10, min_profit_factor=1.0,
                        min_count_test=10, feature_columns=cols)
disc.discover(feats)
gen = StrategyGenerator(disc, risk=RiskParams(sl_atr=2, tp_atr=3, be_atr=1, trail_atr=1.5))
strategies = gen.build()
print('Strategie:', len(strategies))

## 2. Export EA

Per ogni strategia generiamo `mql4/<nome>.mq4` e `mql5/<nome>.mq5`. Se non ci sono strategie stabili, costruiamo un EA dimostrativo da un cluster qualsiasi per mostrare l'output.

In [ ]:
from trading_ai.pattern_discovery.clustering import FeatureClusterer
from trading_ai.strategy_generator import Strategy

eag = EAGenerator()
if strategies:
    paths = eag.export(strategies[0])
else:
    clu = FeatureClusterer(n_clusters=8, feature_columns=cols).fit(feats)
    demo = Strategy(name='DEMO_LONG', cluster_id=0, direction=1, clusterer=clu,
                    risk=RiskParams(sl_atr=2, tp_atr=3))
    paths = eag.export(demo)
print('Generati:', paths['mql4'].name, '|', paths['mql5'].name)

In [ ]:
# Anteprima delle prime righe del file MQL5 generato.
print('\n'.join(paths['mql5'].read_text().splitlines()[:40]))

## ✅ Modulo 6 verificato

Export MQL4/MQL5 con modello embeddato, codice modulare e commentato, controllo delle feature esportabili. Test: `pytest tests/test_ea_generator.py` (7/7).

> La compilazione finale va fatta in MetaEditor (non disponibile in CI): il codice segue le API standard MQL4/MQL5. Copia i file dalle cartelle `mql4/` e `mql5/` nelle rispettive `Experts/` del terminale.

**Prossimo:** Modulo 7 — AI Feedback (analisi degli errori, ottimizzazione automatica, versioning delle strategie).